In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image

In [ ]:
dataset_path = r"D:\Downloads\chest_xray"

In [ ]:
train_path = os.path.join(dataset_path, "train")
val_path = os.path.join(dataset_path, "val")
test_path = os.path.join(dataset_path, "test")

In [ ]:
print("Train:", train_path)
print("Validation:", val_path)
print("Test:", test_path)

In [ ]:
print("Train folder exists:", os.path.exists(train_path))
print("Validation folder exists:", os.path.exists(val_path))
print("Test folder exists:", os.path.exists(test_path))

print("\nTrain classes:")
print(os.listdir(train_path))

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode="binary"
)

In [ ]:
validation_generator = test_datagen.flow_from_directory(
    val_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode="binary"
)

In [ ]:
test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

print("Class mapping:", train_generator.class_indices)

In [ ]:
model = Sequential([
    
    Conv2D(32, (3, 3), activation="relu",
           input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.5),

    Dense(1, activation="sigmoid")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_generator)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print("Test Accuracy (%):", round(test_accuracy * 100, 2))

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

In [ ]:
model.save("pneumonia_cnn_model.keras")

print("Model saved successfully!")

In [ ]:
image_path = r"D:\Downloads\chest_xray\test\PNEUMONIA\person1_bacteria_1.jpeg"

In [ ]:
img = image.load_img(
    image_path,
    target_size=(150, 150)
)

img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)

img_array = img_array / 255.0

prediction = model.predict(img_array)

print("Prediction value:", prediction[0][0])

In [ ]:
if prediction[0][0] >= 0.5:
    print("Prediction: PNEUMONIA")
else:
    print("Prediction: NORMAL")

In [ ]:
plt.figure(figsize=(6, 6))

plt.imshow(image.load_img(image_path), cmap="gray")
plt.axis("off")

if prediction[0][0] >= 0.5:
    result = "PNEUMONIA"
else:
    result = "NORMAL"

plt.title("Prediction: " + result)
plt.show()